# 🧠🤖 第4周-Day2：向量检索：Embedding 与相似度计算

## 📚 学习目标
今天我们要深入理解向量检索的核心技术——Embedding和相似度计算，这是RAG系统的基础！

### 🎯 主要内容
1. 什么是Embedding？
2. 向量空间的概念
3. 相似度计算方法
4. 实战代码演示
5. 业务应用场景

## 🔍 什么是Embedding？

Embedding（嵌入）是一种将离散符号（如文字、图片）转换为连续向量的技术。想象一下：

🎯 **核心思想**：将语义相关的文本映射到向量空间中相近的位置

📊 **可视化理解**：
- "机器学习"和"深度学习" → 距离很近
- "猫咪"和"狗" → 中等距离
- "苹果"和"手机" → 距离较远

💡 **关键特性**：
- 固定维度的向量（通常768维、1024维等）
- 语义相近 → 向量距离相近
- 支持数学运算（相似度计算、向量运算）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Droid Sans Fallback', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建一些示例向量（模拟embedding）
vectors = {
    "机器学习": np.array([0.8, 0.6, 0.2, 0.1]),
    "深度学习": np.array([0.7, 0.5, 0.3, 0.2]),
    "猫咪": np.array([0.2, 0.8, 0.1, 0.9]),
    "狗": np.array([0.3, 0.7, 0.2, 0.8]),
    "苹果": np.array([0.9, 0.1, 0.8, 0.2]),
    "手机": np.array([0.1, 0.9, 0.3, 0.7])
}

print("📊 示例向量表示：")
for key, vec in vectors.items():
    print(f"{key}: {vec}")

## 📐 向量空间与相似度计算

### 🔸 常用相似度计算方法

1. **余弦相似度** (Cosine Similarity)
   - 计算向量夹角的余弦值
   - 值域：[-1, 1]，越接近1越相似
   - 适合文本语义相似度

2. **欧几里得距离** (Euclidean Distance)
   - 计算向量之间的直线距离
   - 值域：[0, ∞)，越小越相似
   - 适合数值型数据

3. **点积** (Dot Product)
   - 向量对应元素相乘后求和
   - 值域：[-∞, ∞]，越大越相似
   - 计算效率高

In [ ]:
# 定义相似度计算函数
def cosine_sim(vec1, vec2):
    """计算余弦相似度"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

def euclidean_dist(vec1, vec2):
    """计算欧几里得距离"""
    return np.linalg.norm(vec1 - vec2)

def dot_product(vec1, vec2):
    """计算点积"""
    return np.dot(vec1, vec2)

# 计算所有向量对的相似度
texts = list(vectors.keys())
cosine_matrix = np.zeros((len(texts), len(texts)))
euclidean_matrix = np.zeros((len(texts), len(texts)))
dot_matrix = np.zeros((len(texts), len(texts)))

for i in range(len(texts)):
    for j in range(len(texts)):
        cosine_matrix[i, j] = cosine_sim(vectors[texts[i]], vectors[texts[j]])
        euclidean_matrix[i, j] = euclidean_dist(vectors[texts[i]], vectors[texts[j]])
        dot_matrix[i, j] = dot_product(vectors[texts[i]], vectors[texts[j]])

print("📊 余弦相似度矩阵：")
print(cosine_matrix)
print("\n📊 欧几里得距离矩阵：")
print(euclidean_matrix)
print("\n📊 点积矩阵：")
print(dot_matrix)

## 📈 可视化相似度结果

让我们用热力图直观展示不同文本之间的相似度关系！

In [ ]:
# 创建热力图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('📈 不同相似度度量方法对比', fontsize=16, fontweight='bold')

# 余弦相似度
sns.heatmap(cosine_matrix, annot=True, cmap='RdYlBu_r', ax=axes[0], 
            xticklabels=texts, yticklabels=texts, vmin=-1, vmax=1)
axes[0].set_title('余弦相似度')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# 欧几里得距离
sns.heatmap(euclidean_matrix, annot=True, cmap='viridis', ax=axes[1], 
            xticklabels=texts, yticklabels=texts)
axes[1].set_title('欧几里得距离')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

# 点积
sns.heatmap(dot_matrix, annot=True, cmap='plasma', ax=axes[2], 
            xticklabels=texts, yticklabels=texts)
axes[2].set_title('点积')
axes[2].tick_params(axis='x', rotation=45)
axes[2].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

print("🎯 关键观察：")
print("• 余弦相似度：语义相似的文本值较高（颜色偏红）")
print("• 欧几里得距离：语义相似的文本值较小（颜色偏深蓝）")
print("• 点积：语义相似的文本值较大（颜色偏黄/白）")

## 🚀 实战：使用真实的Embedding模型

现在让我们使用HuggingFace的Sentence Transformers来生成真实的embedding！

In [ ]:
# 安装必要的包
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# 安装sentence-transformers
try:
    from sentence_transformers import SentenceTransformer, util
    print("✅ Sentence Transformers 已安装")
except ImportError:
    print("🔄 正在安装 Sentence Transformers...")
    install_package('sentence-transformers')
    from sentence_transformers import SentenceTransformer, util

In [ ]:
# 加载预训练模型
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'  # 多语言模型
model = SentenceTransformer(model_name)

# 要编码的文本
sentences = [
    "机器学习是人工智能的重要分支",
    "深度学习属于机器学习的子集",
    "我喜欢喝糖水",
    "美华糖水店的芋圆很好吃",
    "Python是一种编程语言",
    "Java是另一种编程语言",
    "我今天想喝奶茶",
    "天气预报说明天会下雨"
]

print("📝 编码文本列表：")
for i, sentence in enumerate(sentences):
    print(f"{i+1}. {sentence}")

# 生成embedding
embeddings = model.encode(sentences)

print(f"\n📊 Embedding形状：{embeddings.shape}")
print(f"每句话都被表示为{embeddings.shape[1]}维向量")

In [ ]:
# 计算余弦相似度
cosine_sim_matrix = util.cos_sim(embeddings)

# 创建相似度矩阵DataFrame
import pandas as pd
sim_df = pd.DataFrame(cosine_sim_matrix.numpy(), 
                     index=sentences, 
                     columns=sentences)

print("📜 实际Embedding的余弦相似度矩阵：")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(sim_df)

# 找出最相似的句子对
max_sim = 0
best_pair = ("", "")

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        if cosine_sim_matrix[i, j] > max_sim:
            max_sim = cosine_sim_matrix[i, j]
            best_pair = (sentences[i], sentences[j])

print(f"\n🎯 最相似的句子对：")
print(f"'{best_pair[0]}' 和 '{best_pair[1]}'")
print(f"相似度：{max_sim:.4f}")

## 🎯 RAG中的向量检索实战

在RAG系统中，向量检索通常包括以下步骤：

1. **Query Encoding** - 将用户查询转换为embedding
2. **Similarity Search** - 在向量数据库中搜索最相似的文档
3. **Ranking** - 按相似度排序并返回最相关的结果

In [ ]:
# 模拟RAG检索过程
def rag_search(query, documents, embeddings, top_k=3):
    """RAG检索函数"""
    # 编码查询
    query_embedding = model.encode([query])
    
    # 计算查询与所有文档的相似度
    query_similarities = util.cos_sim(query_embedding, embeddings)[0]
    
    # 获取top_k个最相似的文档
    top_indices = np.argsort(query_similarities)[-top_k:][::-1]
    
    # 返回结果
    results = []
    for idx in top_indices:
        results.append({
            'document': documents[idx],
            'similarity': query_similarities[idx].item(),
            'index': idx
        })
    
    return results

# 测试查询
test_query = "我想喝糖水，有什么推荐的吗？"

print(f"🔍 用户查询：{test_query}")
print("\n📋 检索到的最相关文档：")
results = rag_search(test_query, sentences, embeddings)

for i, result in enumerate(results, 1):
    print(f"{i}. 相似度：{result['similarity']:.4f}")
    print(f"   文档：{result['document']}")
    print()

## 💡 业务应用场景：美华糖水店

让我们看看embedding技术如何应用到美华糖水店的业务中：

In [ ]:
# 美华糖水店的菜单数据
menu_items = [
    "招牌芋圆奶茶，香浓奶茶配Q弹芋圆",
    "杨枝甘露，芒果西柚配西米露",
    "红豆冰，传统经典红豆冰沙",
    "绿豆沙，清热解暑绿豆沙",
    "珍珠奶茶，经典珍珠奶茶",
    "柠檬茶，清爽柠檬茶",
    "西瓜汁，新鲜西瓜汁",
    "冰淇淋，香草冰淇淋",
    "布丁，鸡蛋布丁",
    "奶盖茶，茶叶配奶盖"
]

# 生成菜单的embedding
menu_embeddings = model.encode(menu_items)

print("🍧 美华糖水店菜单：")
for i, item in enumerate(menu_items):
    print(f"{i+1}. {item}")

# 模拟不同的顾客查询
customer_queries = [
    "我想喝甜的",
    "今天天气热，想要解暑的",
    "我喜欢喝茶",
    "推荐一个招牌的",
 "我想喝奶茶类的"
]

print("\n" + "="*60)
print("🎯 智能推荐系统演示：")
print("="*60)

for query in customer_queries:
    print(f"\n👤 顾客：{query}")
    
    # 检索推荐
    query_embedding = model.encode([query])
    similarities = util.cos_sim(query_embedding, menu_embeddings)[0]
    
    # 获取top 3推荐
    top_indices = np.argsort(similarities)[-3:][::-1]
    
    print("💡 推荐结果：")
    for idx in top_indices:
        print(f"   • {menu_items[idx]} (相似度: {similarities[idx]:.3f})")

## 🧩 课堂练习答案

1. "机器学习"和"深度学习"的embedding相似度预计是？**高**（都是AI领域相关术语）
2. "猫咪"和"狗"的embedding相似度预计是？**中等**（都是动物但不同种类）
3. 余弦相似度值域范围是？**[-1, 1]**，1表示完全相同，-1表示完全相反

## 📝 课后测试答案

❶ B. 余弦相似度（最适合文本语义相似度）

❷ A. 完全相同（余弦相似度为1表示向量方向完全一致）

❸ C. 很高（>0.7）（"Hello"和"Hi"是近义词，语义高度相似）

❸ B. 理解语义（向量检索能理解语义关系，不依赖关键词）

❷ C. 蛇类动物（语义最不相关）

## 🎯 下节预告
明天我们将学习高级RAG技术，包括混合检索、重排序和查询改写，让我们的检索系统更加强大！

In [ ]:
# 总结：向量检索的关键要点
print("🎯 第4周-Day2 学习总结：")
print("=" * 50)
print("✅ Embedding技术将文本转换为高维向量")
print("✅ 向量空间中语义相近的文本距离更近")
print("✅ 余弦相似度是最常用的文本相似度度量")
print("✅ 向量检索实现语义理解，超越关键词匹配")
print("✅ 在RAG系统中，向量检索是核心组成部分")
print("✅ 实际应用：智能推荐、搜索问答、文档检索")
print("=" * 50)
print("\n🚀 准备好迎接明天的高级RAG技术了吗？")